# Example: Advanced constraints in static inverse free-boundary equilibrium calculations

---

In this example notebook, we demonstrate some of the more advanced types of constraints and techniques that can be employed in the inverse solver. 

#### Instantiate the objects

As before, we start by instantiating the machine, equilibrium, profiles, and solver objects. 

In [ ]:
# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path="../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

from freegsnke import equilibrium_update
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=0.1, Rmax=2.0,   # radial range
    Zmin=-2.2, Zmax=2.2,  # vertical range
    nx=65,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    # psi=plasma_psi
)

# initialise the profiles
from freegsnke.jtor_update import ConstrainPaxisIp
profiles = ConstrainPaxisIp(
    eq=eq,        # equilibrium object
    paxis=8e3,    # profile object
    Ip=6e5,       # plasma current
    fvac=0.5,     # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2   # profile function parameter
)

from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)    

In [ ]:
# as before, let's fix the Solenoid current
eq.tokamak.set_coil_current('Solenoid', 5000)
eq.tokamak['Solenoid'].control = False  # ensures the current in the Solenoid is fixed

### Feature 1: (Normalised) poloidal flux constraints at specific locations

In addition to the `null points`, `isoflux set`, and `coil_current_limits` constraints introduced in the previous notebook, additional methods are available for more advanced control of the inverse problem.

We can also constrain:

- **Flux values (`psi_vals`)**  
  If the flux is known at a location $(R_j, Z_j)$, we can impose
  $$
  \psi(R_j, Z_j) = \psi_j^{\text{target}}.
  $$
  More generally, one could prescribe $\psi(R,Z)$ over a region (e.g. a full flux map). Flux values are typically difficult to estimate a priori to simulation and so the following constraint is often more useful. 

- **Normalised flux values (`psi_norm_limits`)**  
  At a location $(R_j, Z_j)$, we can impose upper (or lower) bound constraints on the normalised flux 
  $$
  \hat{\psi}(R,Z) = \frac{\psi(R,Z) - \psi_{axis}}{\psi_{boundary} - \psi_{axis}},
  $$ 
  such that 
  $$
  \hat{\psi}(R_j, Z_j) \leq \hat{\psi}_j^{\text{target}}.
  $$
  or
  $$
  \hat{\psi}(R_j, Z_j) \geq \hat{\psi}_j^{\text{target}}.
  $$
  Note that $\psi_{axis}$ and $\psi_{boundary}$ are the values of the flux on the magnetic axis and plasma boundary, respectively.
  As we'll see, this can be useful for setting explicit constraints on flux behaviour near the wall if there are certain safety limits to adhere to.

Once again, we stress that users should be mindful of specifying too many constraints that may conflict with one another during an inverse solve. For example, a given normalised $\psi$ constraint may be impossible/difficult to achieve given other isoflux constraints or coil current limits. More generally, specifying too many constraints could make the problem ill-posed and reduce the overall quality of the final solution. 

Recall that if you find that the solver is violating the limit (inequality) constraints (coil limits and normalised $\psi$), you should look to reduce the number of constraints or try modifying the `mu_*` parameters. These parameters control how much a violation of the constraint is penalised in the solver, and increasing this penalty (by increasing `mu_*`) will increase the likelihood the constraint is adequately satisfied. Again, it may also be that the constraint is impossible to satisfy given your other constraints.

In the following, we will show how to use the normalised flux constraints to control the normalised flux on the divertor nose.

In [ ]:
import numpy as np 
from freegsnke.inverse import Inverse_optimizer

Rx = 0.6      # X-point radius
Zx = 1.1      # X-point height
Rout = 1.4    # outboard midplane radius
Rin = 0.34    # inboard midplane radius

# set desired null_points locations (this can include X-point and O-point locations)
null_points = [[Rx, Rx], [Zx, -Zx]]

# Set desired isoflux constraints with format 
# isoflux_set = [isoflux_0, isoflux_1 ... ] 
# with each isoflux_i = [R_coords, Z_coords, weights]
isoflux_set = np.array([
    [
        [Rx, Rx, Rin, Rout], 
        [Zx, -Zx, 0., 0.],
    ]
])

# set the coil current limits (upper and lower)
# coil ordering in this case: PX,  D1,  D2,  D3,  Dp,  D5,  D6,  D7,  P4,  P5,  P6
coil_current_limits = [
    [5e3, 9e3, 9e3, 7e3, 7e3, 5e3, 4e3, 5e3, 0.0, 0.0, None],
    [-5e3, -9e3, -9e3, -7e3, -7e3, -5e3, -4e3, -5e3, -10e3, -10e3, None]
]

# normalised psi constraints set with format:
# [R, Z, psiN, 1] --> ψ̂(R,Z) ≥ psiN  (the +1 or -1 defines ≥ or ≤)
# [R, Z, psiN, -1] --> ψ̂(R,Z) ≤ psiN
psi_norm_limits = [
    [0.82, -1.55, 1.2, 1],
    [0.82, -1.55, 1.3, -1],
]

# instantiate the freegsnke constrain object
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    coil_current_limits=coil_current_limits,
    psi_norm_limits=psi_norm_limits,
    mu_psi_norm = 1e7, # penalise how much the normalised psi constraint is violated (default 1e6)
)



We solve, as in the previous example, by passing the equilibrium, profiles, and constraints to the static solver. Again, we apply regularisation to encourage a solution with low coil currents.

In [ ]:
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=constrain, 
                     target_relative_tolerance=1e-6,
                     target_relative_psit_update=1e-3,
                     verbose=True, # print output
                     l2_reg=np.array([1e-12]*10+[1e-6]), 
                     )

In [ ]:
import matplotlib.pyplot as plt

# plot the resulting equilibria 
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=True)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

We can now check that the normalised $\psi$ constraint has indeed been satisfied!

In [ ]:
for psi_con in psi_norm_limits:
    sign = ">=" if psi_con[3] >= 0 else "<="
    print(f"Psi norm at ({psi_con[0]}, {psi_con[1]}) = {eq.psiNRZ(psi_con[0], psi_con[1]):.3f} ({sign} {psi_con[2]})")

### Feature 2: Weighting constraints within the solver

It is possible to specify **weights** on the different types of constraints within the inverse solver, enabling the solver to prioritise certain constraints over others.

There are two ways of doing this:

1. **Weighting different types of constraints relative to one another:**

   Different classes of constraint can be weighted relative to each other (e.g. null points, isoflux, and psi constraints). This allows the solver to be instructed that one type of constraint is more important than others. For example, a user may want null point constraints to be strictly satisfied, but may be less concerned about the isoflux set being met with the same strictness.

2. **Weighting individual isoflux constraints within a set:**

   Individual isoflux constraints within a set can optionally be weighted relative to one another, instructing the solver to prioritise satisfying some subset over others.
   
   For example, consider a set of three isoflux constraints: one at the X-point, one at the outer midplane, and one at a desired strike point. The strike point constraint could be assigned a lower weight to allow the solver to find a solution with a strike point close to — but not necessarily exactly on — the target location.

#### Option 1: weighting different types of constraints

Here, we'll show how to weight the null point constraints **more** than the isoflux constraints. This solver then prioritises fitting the null points exactly and the isoflux points less so. 

In [ ]:
# core constraints
Rx = 0.55     # X-point radius
Zx = 1.2      # X-point height
Rout = 1.4    # outboard midplane radius
Rin = 0.34    # inboard midplane radius

# set desired null_points locations (this can include X-point and O-point locations)
null_points = [[Rx, Rx], [Zx, -Zx]]

# set desired isoflux constraints with format 
# isoflux_set = [isoflux_0, isoflux_1 ... ] 
# with each isoflux_i = [R_coords, Z_coords]
isoflux_set = np.array([
    [
        [Rx, Rx, Rin, Rout, 0.75, 1.0, 0.75, 1.0], 
        [Zx, -Zx, 0.0, 0.0, -1.6, -2.0, 1.6, 2.0],
    ]
])

# instantiate the freegsnke constrain object
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    coil_current_limits=coil_current_limits,
    weight_isoflux=0.2, # <-- weight the isofluxes less than the null points (default 1.0)
    weight_nulls=1.0,   # <-- weight the null points more than the null points (default 1.0)
    # weight_psi=1.0    # <-- weight for psi values not used here
)



In [ ]:
# solve!
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=constrain, 
                     target_relative_tolerance=1e-6,
                     target_relative_psit_update=1e-3,
                     verbose=True,
                     l2_reg=np.array([1e-12]*10+[1e-6]), 
                     )

Notice how the null points constraints are satisfied perfectly while the isoflux constraints are "less so" (especially in the divertor region).

In [ ]:
# plot the resulting equilibria 
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=True)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

#### Option 2: weighting different isoflux constraints

Here, we'll show how to weight individual isoflux constraints differently from one another. For example, here we want to prioritise the core shape over the exact location of the strikepoint. 

In [ ]:
# core constraints
Rx = 0.55     # X-point radius
Zx = 1.2      # X-point height
Rout = 1.4    # outboard midplane radius
Rin = 0.34    # inboard midplane radius

# set desired null_points locations (this can include X-point and O-point locations)
null_points = [[Rx, Rx], [Zx, -Zx]]

# set desired isoflux constraints with format 
# isoflux_set = [isoflux_0, isoflux_1 ... ] 
# with each isoflux_i = [R_coords, Z_coords, weights]
isoflux_set = np.array([
    [
        [Rx, Rx, Rin, Rout, 1.0, 1.0, 0.75, 1.0, 0.75, 1.0], 
        [Zx, -Zx, 0.0, 0.0, -0.9, 0.9,  -1.6, -2.0, 1.6, 2.0],
        [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.1, 1.0, 0.1],
    ]
])

# instantiate the freegsnke constrain object
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    coil_current_limits=coil_current_limits,
)

In [ ]:
# solve!
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=constrain, 
                     target_relative_tolerance=1e-6,
                     target_relative_psit_update=1e-3,
                     verbose=True, # print output
                     l2_reg=np.array([1e-12]*10+[1e-6]), 
                     )

Notice how the strikepoint isoflux constraint is no longer satisfied.

In [ ]:
# plot the resulting equilibria 
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=True)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

### Feature 3: second-order magnetic null constraints (i.e. generating snowflake divertors)

Now we'll try to use the more complex constraints to form second order nulls (using the `null_points_2nd_order` parameter) and a snowflake divertor geometry. 


In some situations, it may be desirable to control the **local structure of the magnetic field** around a null point. This is achieved by enforcing **second-order null point constraints**, which additionally constrain the *spatial derivatives* of the magnetic field.

Recall that the poloidal magnetic field is related to the poloidal flux $\psi$ via

$$
B_R = -\frac{1}{R} \frac{\partial \psi}{\partial Z}, 
\qquad
B_Z = \frac{1}{R} \frac{\partial \psi}{\partial R}.
$$

A standard null point $(R_X, Z_X)$ satisfies

$$
B_R(R_X, Z_X) = B_Z(R_X, Z_X) = 0.
$$

Second-order null point constraints extend this by additionally imposing conditions on the **Jacobian of the magnetic field**, i.e. its first spatial derivatives:

$$
\frac{\partial B_R}{\partial R}, \quad
\frac{\partial B_R}{\partial Z}, \quad
\frac{\partial B_Z}{\partial R}, \quad
\frac{\partial B_Z}{\partial Z}.
$$

In FreeGSNKE, the following constraints are enforced at each second-order null point:

$$
B_R = 0, \qquad B_Z = 0,
$$

$$
\frac{\partial B_R}{\partial R} = 0, \qquad
\frac{\partial B_Z}{\partial Z} = 0, \qquad
\frac{\partial B_R}{\partial Z} = 0, \qquad
\frac{\partial B_Z}{\partial R} = 0.
$$

This results in a total of **six constraints per null point** so be aware that:
- imposing too many (second-order) constraints can easily **overconstrain** the system.
- in practice, they are most useful when combined with a limited number of other constraints (e.g. isoflux sets).

Here, we'll use one second-order null constraint in combination with a few isoflux constraints. 

Also note how we increase `constrain.mu_coils` as snowflake geometries tend to demand a lot from the PF coils. We need to make sure we don't violate their limits. 

But first let's reset the equilibrium and profile objects (cranking up the resolution slightly).

In [ ]:
# we are modifying the control, build a new tokamak
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path="../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

# a new eq object resets the coil currents and equilibrium
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=0.1, Rmax=2.0,   # radial range
    Zmin=-2.2, Zmax=2.2,  # vertical range
    nx=129,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
)

# reset the profiles
profiles = ConstrainPaxisIp(
    eq=eq,        # equilibrium object
    paxis=8e3,    # profile object
    Ip=6e5,       # plasma current
    fvac=0.5,     # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2   # profile function parameter
)

# re-initialise the solver object (due to new grid resolution)
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)    

# first we specify some alternative constraints
Rout = 1.32  # outboard midplane radius
Rin = 0.3   # inboard midplane radius

# locations of X-points (this time they will be second order nulls!)
Rx = 0.65
Zx = 1.2

# second order null constraints to form the snowflake
null_points_2nd_order = [[Rx], [Zx]]

# isoflux constraints (no divertor constraints this time)
isoflux_set = np.array([[[Rx, Rin, Rout], [Zx, 0.,0.]]])

# starting guess for the Solenoid current
# eq.tokamak.set_coil_current('Solenoid', 5000)
# eq.tokamak['Solenoid'].control = True

# also make it perfectly up/down symmetric
eq.tokamak.set_coil_current('P6', 0)
eq.tokamak['P6'].control = False

coil_current_limits = [
# upper limits...
#    P1, PX,   D1,  D2,  D3,   Dp,   D5,  D6,   D7,   P4,  P5
    [1e4, 6e3, 9e3, 9e3, 7e3, 7e3, 5e3, 3.5e3, 5e3, 0, 0],
# lower limits...
#    P1, PX,    D1,    D2,    D3,    Dp,    D5,   D6,    D7,    P4,    P5
    [-1e4, -6e3, -9e3, -9e3, -7e3, -7e3, -5e3, -3.5e3, -5e3, -1.2e4, -1.2e4]
]

# pass the magnetic constraints to a new constrain object
constrain = Inverse_optimizer(
    null_points_2nd_order=null_points_2nd_order,
    isoflux_set=isoflux_set,
    coil_current_limits=coil_current_limits,
)

# increase this to ensure coil currents stay with limits
constrain.mu_coils = 5e7

# carry out the solve
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=constrain, 
                     target_relative_tolerance=1e-6,
                     target_relative_psit_update=1e-3,
                     verbose=True,
                     l2_reg=np.array([1e-12]*11),
                     )

In [ ]:
# plot the resulting equilibrium
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=100)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=True)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

Zoom in to see the snowflake divertor geometry!

In [ ]:
# upper X-point
fig1, ax1 = plt.subplots(1, 1, figsize=(8, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=False)
ax1.set_xlim(0.15, 1.15)
ax1.set_ylim(0.75, 1.75)
plt.tight_layout()

# lower X-point
fig1, ax1 = plt.subplots(1, 1, figsize=(8, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=False)
ax1.set_xlim(0.15, 1.15)
ax1.set_ylim(-1.75, -0.75)
plt.tight_layout()

Just double check that the coil currents stayed within the limits!

In [ ]:
# storage
coil_names = []
active_currents = []
min_limits = []
max_limits = []

for coil_name, coil_current, ul, ll in zip(
    eq.tokamak.coils_dict,
    eq.tokamak.getCurrentsVec()[:12],
    coil_current_limits[0] + [None],   # upper limits
    coil_current_limits[1] + [None],   # lower limits
):
    coil_names.append(coil_name)
    active_currents.append(coil_current)
    min_limits.append(ll)
    max_limits.append(ul)

    # skip coils without limits
    if ul is not None or ll is not None:
        if (coil_current >= ll) & (coil_current <= ul):
            within_limit = True
        else:
            within_limit = False
    else:
        within_limit = None

    # print
    print(f"{coil_name}: {coil_current:.2f} --> [{ll},{ul}] [A] --> limits met = {within_limit}")

# convert to numpy
active_currents = np.array(active_currents)
min_limits = np.array(min_limits)
max_limits = np.array(max_limits)

In [ ]:
# VISUALISE HOW CLOSE THEY ARE TO THE LIMITS (normalised)

# normalized to [-1, 1] scale (relative to min/max)
min_limits[min_limits==None] = np.inf
max_limits[max_limits==None] = np.inf
norm_current = 2 * (active_currents - min_limits) / (max_limits - min_limits) - 1

# plot
plt.figure(figsize=(12, 5), dpi=70)
x = np.arange(len(coil_names))

# line plot
plt.plot(x, norm_current, marker='x', linestyle='-', color='red', label="Current")
plt.axhline(-1, color='k', linestyle='-', linewidth=1, label="Min. limit")
plt.axhline(1, color='k', linestyle='-', linewidth=1, label="Max. limit")

# formatting
plt.xticks(x, coil_names, rotation=45, ha='right')
plt.ylabel('Coil name [Amps]')
plt.ylabel('Normalised current')
plt.title('Coil current proximity to limits (normalised)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()
